In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

In [2]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._

import org.dataprov.dp.sparkdataprovenance.ProvenanceApi._
import org.dataprov.dp.sparkdataprovenance.LogicalPlanWithProvenance
import org.dataprov.dp.sparkdataprovenance.SparkProvenanceExtension
import org.dataprov.dp.sparkdataprovenance.FullWhyProvenanceBuilder
import org.dataprov.dp.sparkdataprovenance.SemiWhyProvenanceBuilder

import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._
import org.dataprov.dp.sparkdataprovenance.ProvenanceApi._
import org.dataprov.dp.sparkdataprovenance.LogicalPlanWithProvenance
import org.dataprov.dp.sparkdataprovenance.SparkProvenanceExtension
import org.dataprov.dp.sparkdataprovenance.FullWhyProvenanceBuilder
import org.dataprov.dp.sparkdataprovenance.SemiWhyProvenanceBuilder

In [4]:
import org.apache.spark.sql.SparkSession

val sparkWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        new SparkProvenanceExtension(
            provenanceBuilder = SemiWhyProvenanceBuilder
        )
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")
sparkWhy.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/17 15:18:33 INFO SparkContext: Running Spark version 4.1.1
26/07/17 15:18:33 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/07/17 15:18:33 INFO SparkContext: Java version 17.0.10+7
26/07/17 15:18:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/17 15:18:33 INFO ResourceUtils: ==============================================================
26/07/17 15:18:33 INFO ResourceUtils: No custom resources configured for spark.driver.
26/07/17 15:18:33 INFO ResourceUtils: ==============================================================
26/07/17 15:18:33 INFO SparkContext: Submitted application: notebook-demo-why-provenance
26/07/17 15:18:33 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/07/17 15:18:33 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/07/17 15:18:33 INFO SecurityManager: Changing

Spark provenance enabled: true


import org.apache.spark.sql.SparkSession
sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@60b12232

In [5]:
// Example of loading a Parquet file and adding provenance

val localPath = "file:///Users/mac-ABALLA16/Downloads/product_hierarchy"
val df = sparkWhy.read.format("parquet").load(localPath)

println(s"Nombre de lignes réelles chargées : ${df.count()}")

val dfWithProv = df.addProvenanceColumn
dfWithProv.show(20, false)


Nombre de lignes réelles chargées : 55526486
+----------+-----------+---------------------------+-------------------+------------------------------------+---------------+-----------------------+-------------+-----------------------------------+--------------+-------------------+-----------------+---------------------------+------------------------------------+
|model_code|family_code|family_name                |sub_department_code|sub_department_name                 |department_code|department_name        |universe_code|universe_name                      |hierarchy_type|validity_start_date|validity_end_date|tech_ingestion_datetime_utc|_provenance_tag                     |
+----------+-----------+---------------------------+-------------------+------------------------------------+---------------+-----------------------+-------------+-----------------------------------+--------------+-------------------+-----------------+---------------------------+------------------------------------+
|

localPath: String = "file:///Users/mac-ABALLA16/Downloads/product_hierarchy"
df: DataFrame = [model_code: bigint, family_code: bigint ... 11 more fields]
dfWithProv: DataFrame = [model_code: bigint, family_code: bigint ... 12 more fields]

In [6]:

def complexPipeline(df: DataFrame): DataFrame = {
  
  val baseDF = df
    .filter(col("model_code").isNotNull && col("validity_start_date").isNotNull)
    .withColumn("start_dt", to_date(col("validity_start_date"), "yyyy-MM-dd"))
    .withColumn("end_dt", to_date(col("validity_end_date"), "yyyy-MM-dd"))
    .select("model_code", "hierarchy_type", "family_code", "family_name", "start_dt", "end_dt")
    .distinct()

  val rankedDF = baseDF.as("r1")
    .join(
      baseDF.as("r2"),
      expr("""
        r1.model_code = r2.model_code 
        AND r1.hierarchy_type = r2.hierarchy_type 
        AND r1.start_dt > r2.start_dt
      """),
      "left"
    )
    .groupBy("r1.model_code", "r1.hierarchy_type", "r1.family_code", "r1.family_name", "r1.start_dt", "r1.end_dt")
    .agg(count("r2.start_dt").as("row_rank")) 

  val anomaliesDF = rankedDF.filter(col("end_dt") >= col("start_dt"))

  val finalDF = anomaliesDF
    .groupBy("family_code", "family_name")
    .agg(countDistinct("model_code").as("nb_modeles_en_conflit"))
    .orderBy(desc("nb_modeles_en_conflit"))

  finalDF
}

val dfResult = complexPipeline(df)
dfResult.show(10, false)

+-----------+--------------------------------------+---------------------+
|family_code|family_name                           |nb_modeles_en_conflit|
+-----------+--------------------------------------+---------------------+
|11957      |MAN ESSENTIALS T-SHIRT & TANK         |14272                |
|11957      |MAN T-SHIRT & TANK ESSENTIALS         |14223                |
|11957      |MAN GYM, PILATES APPAREL              |14173                |
|11957      |MAN FIT ACTIVE LIGHT MAN TS SHORT TANK|14172                |
|11957      |MAN PILATES, SOFT GYM APPAREL         |14127                |
|11957      |MAN T SHIRT SHORT                     |14121                |
|11957      |MAN T-SHIRT & SHORT DAILY             |13830                |
|10789      |WOMAN CARDIO & STRENGTH               |13044                |
|10789      |WOMAN CARDIO & STRENGTH PERF          |12738                |
|11956      |WOMAN T SHIRT LEGGING SHORT           |12578                |
+-----------+------------

defined function complexPipeline
dfResult: DataFrame = [family_code: bigint, family_name: string ... 1 more field]

In [10]:
val aggregate = dfWithProv.select("model_code", "family_name", "department_name","sub_department_name").groupBy("sub_department_name", "model_code").agg(count("*").as("count"))
//aggregate.show(10,false)

val finalRows = aggregate.filter(col("model_code") === "1037865" || col("model_code") === "1127317").select("_provenance_tag")
//finalRows.show(20, false)

//sparkWhy.conf.set("spark.provenance.enabled", "false")

withProvenanceDisabled(sparkWhy) {
  val finalTags = finalRows
    .select(explode((col("_provenance_tag"))).as("_provenance_tag"))
    .distinct()

   val initialRows = dfWithProv
    .join(finalTags, Seq("_provenance_tag"), "left_semi")

   initialRows.filter(col("sub_department_code") === 1962).show(20, false)
}

+------------------------------------+----------+-----------+------------------+-------------------+--------------------+---------------+---------------------------+-------------+--------------------------+--------------+-------------------+-----------------+---------------------------+
|_provenance_tag                     |model_code|family_code|family_name       |sub_department_code|sub_department_name |department_code|department_name            |universe_code|universe_name             |hierarchy_type|validity_start_date|validity_end_date|tech_ingestion_datetime_utc|
+------------------------------------+----------+-----------+------------------+-------------------+--------------------+---------------+---------------------------+-------------+--------------------------+--------------+-------------------+-----------------+---------------------------+
|51969df3-844d-4789-99bf-56244c9563bf|1127317   |10870      |RIFFLE AMMUNITIONS|1962               |Big game ammunitions|13             

aggregate: DataFrame = [sub_department_name: string, model_code: bigint ... 2 more fields]
finalRows: DataFrame = [_provenance_tag: array<string>]

In [ ]:
import org.dataprov.dp.sparkdataprovenance.ProvenanceExtractor

val rowsToDebug = aggregate.filter(col("model_code") === "1037865" || col("model_code") === "1127317")

withProvenanceDisabled(sparkWhy) {
  val Seq(initialRowsFromFramework) = ProvenanceExtractor.extractMinimalDatasets(
    finalDF = rowsToDebug,
    sourceDFs = Seq(dfWithProv),
    provenanceColName = "_provenance_tag"
  )

  initialRowsFromFramework.filter(col("sub_department_code") === 1962).show(20, false)
}

+----------+-----------+------------------+-------------------+--------------------+---------------+---------------------------+-------------+--------------------------+--------------+-------------------+-----------------+---------------------------+------------------------------------+
|model_code|family_code|family_name       |sub_department_code|sub_department_name |department_code|department_name            |universe_code|universe_name             |hierarchy_type|validity_start_date|validity_end_date|tech_ingestion_datetime_utc|_provenance_tag                     |
+----------+-----------+------------------+-------------------+--------------------+---------------+---------------------------+-------------+--------------------------+--------------+-------------------+-----------------+---------------------------+------------------------------------+
|1127317   |10870      |RIFFLE AMMUNITIONS|1962               |Big game ammunitions|13             |WILDLIFE WATCHING / HUNTING|7       

import org.dataprov.dp.sparkdataprovenance.ProvenanceExtractor
rowsToDebug: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [sub_department_name: string, model_code: bigint ... 2 more fields]

In [8]:
val distinct = dfWithProv.select("family_name").distinct()
distinct.show(false)

+--------------------------------------+--------------------------------------+
|family_name                           |_provenance_tag                       |
+--------------------------------------+--------------------------------------+
|'PULL-APART' POLE FISHING RODS, ACCESS|[3641baf4-f9af-45d4-801e-4f0ee5c188a3]|
|(!) EMPTY EXPO BACKBOARDS             |[8fd3d777-6cb4-4ede-b173-3be63ba20b02]|
|(!) EXPO BACKBOARDS                   |[4944ad4a-edcf-4b7c-a17c-67cc6fd19377]|
|0L Replica                            |[d2da900f-134b-441d-8b2a-3b56cbe104f0]|
|10L TO 30L NATURE HIKING BACKPACKS    |[10f90e79-66ce-4cd7-ae46-5af1b631d120]|
|11 FOOTBALL BALLS                     |[04998d59-78d5-4851-801a-fcfd3326bb1d]|
|12"/14" Bikes (3-5 years)             |[3db2cf71-3fa2-417b-b44b-a833a24bdbe3]|
|16 INCHES BIKES (4-6 YEARS)           |[9a8de024-def5-4e04-9e18-41526143d532]|
|16I                                   |[1d10f26a-cf01-4131-8aa6-6aa3b18783fd]|
|20 GA CARTRIDGES/SLUGS                |

distinct: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [family_name: string, _provenance_tag: array<string>]